### Get imports

In [1]:
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import defaultdict

from tqdm import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score

from utils.utils import convert_to_gpu, get_neighbor_sampler
from utils.DataLoader import get_link_prediction_data
from utils.EarlyStopping import EarlyStopping
from models.DyGFormer import DyGFormer
from models.modules import MergeLayer

from explain.TempME.tempme import TempMEDyGFormer
from explain.TempME.utils import (
    NeighborFinder, 
    compute_teacher_predictions, 
    train_tempme, 
    get_explanation_parameters,
    get_explanations,
    assess_explanations
)

/Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load in the GNN Model

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [3]:
DATASET_NAME = "wikipedia"
EXAMPLES = 5000
BATCH_SIZE = 1000
NUM_WALKS = 2
LEARNING_RATE = 1e-3
BETA = 0.5
PRIOR_P = 0.3
N_DEGREE = 20

In [4]:
(
    node_raw_features,
    edge_raw_features,
    full_data,
    train_data,
    val_data,
    test_data,
    _,
    _
) = get_link_prediction_data(
    dataset_name=DATASET_NAME,
    val_ratio=0.15,
    test_ratio=0.15
)

The dataset has 157474 interactions, involving 9227 different nodes
The training dataset has 79202 interactions, involving 5904 different nodes
The validation dataset has 23621 interactions, involving 3256 different nodes
The test dataset has 23621 interactions, involving 3564 different nodes
The new node validation dataset has 11742 interactions, involving 2134 different nodes
The new node test dataset has 11765 interactions, involving 2482 different nodes
922 nodes were used for the inductive testing, i.e. are never seen during training


In [5]:
full_neighbor_sampler = get_neighbor_sampler(
    data=full_data,
    sample_neighbor_strategy="recent",
    time_scaling_factor=1e-6,
    seed=1
)

In [6]:
# using the parameters that I trained the model on
dynamic_backbone = DyGFormer(
    node_raw_features=node_raw_features,
    edge_raw_features=edge_raw_features,
    neighbor_sampler=full_neighbor_sampler,
    time_feat_dim=100,
    channel_embedding_dim=50,
    patch_size=2,
    num_layers=2,
    num_heads=2,
    dropout=0.1,
    max_input_sequence_length=64,
    device="cpu"
)

In [7]:
dynamic_backbone.n_feat_th = dynamic_backbone.node_raw_features
dynamic_backbone.e_feat_th = dynamic_backbone.edge_raw_features

In [8]:
link_predictor = MergeLayer(
    input_dim1=node_raw_features.shape[1],
    input_dim2=node_raw_features.shape[1],
    hidden_dim=node_raw_features.shape[1],
    output_dim=1
)

In [9]:
model = nn.Sequential(dynamic_backbone, link_predictor)

In [10]:
load_model_folder = f"./saved_models/DyGFormer/{DATASET_NAME}/DyGFormer_seed1"
early_stopping = EarlyStopping(
    patience=0,
    save_model_folder=load_model_folder,
    save_model_name="DyGFormer_seed1",
    logger=logger,
    model_name="DyGFormer"
)
early_stopping.load_checkpoint(model, map_location="cpu")

INFO:root:load model ./saved_models/DyGFormer/wikipedia/DyGFormer_seed1/DyGFormer_seed1.pkl


In [11]:
model = convert_to_gpu(model, device="cpu")
model.eval()

Sequential(
  (0): DyGFormer(
    (time_encoder): TimeEncoder(
      (w): Linear(in_features=1, out_features=100, bias=True)
    )
    (neighbor_co_occurrence_encoder): NeighborCooccurrenceEncoder(
      (neighbor_co_occurrence_encode_layer): Sequential(
        (0): Linear(in_features=1, out_features=50, bias=True)
        (1): ReLU()
        (2): Linear(in_features=50, out_features=50, bias=True)
      )
    )
    (projection_layer): ModuleDict(
      (node): Linear(in_features=344, out_features=50, bias=True)
      (edge): Linear(in_features=344, out_features=50, bias=True)
      (time): Linear(in_features=200, out_features=50, bias=True)
      (neighbor_co_occurrence): Linear(in_features=100, out_features=50, bias=True)
    )
    (transformers): ModuleList(
      (0-1): 2 x TransformerEncoder(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
        )
        (dropout): Dropout(p=

In [12]:
device = next(model.parameters()).device

### Load in the Explainer Model

In [13]:
# build an adjaceny list
adj_list = [[] for _ in range(full_data.num_unique_nodes + 1)]
for src, dst, edge_id, ts in zip(
    full_data.src_node_ids, full_data.dst_node_ids, full_data.edge_ids, full_data.node_interact_times
):
    adj_list[src].append((dst, edge_id, ts))
    adj_list[dst].append((src, edge_id, ts))

In [14]:
neighbor_finder = NeighborFinder(adj_list=adj_list)

In [15]:
tempme = TempMEDyGFormer(
    base=dynamic_backbone,
    base_model_type="dygformer",
    data=DATASET_NAME,
    out_dim=1,
    hid_dim=node_raw_features.shape[1],
    device=device
)

In [16]:
np.random.seed(42)
test_indices = np.random.choice(
    len(test_data.src_node_ids),
    size=min(EXAMPLES, len(test_data.src_node_ids)),
    replace=False
)

In [17]:
example_data = compute_teacher_predictions(
    model=model,
    data=test_data,
    indices=test_indices
)

INFO:root:Computing teacher predictions for 5000 examples...
100%|██████████| 5000/5000 [00:17<00:00, 292.46it/s]
INFO:root:
        Teacher labels: 4480.0 postive, 
        520.0 negative
    


In [18]:
tempme.train()
optimizer = torch.optim.Adam(
    tempme.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0
)
criterion = nn.BCEWithLogitsLoss() 

In [19]:
train_tempme(
    data=example_data,
    batch_size=BATCH_SIZE,
    num_walks=NUM_WALKS,
    prior_p=PRIOR_P,
    beta=BETA,
    neighbor_finder=neighbor_finder,
    tempme=tempme,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

Training TempME: 100%|██████████| 10/10 [01:57<00:00, 11.77s/it]
INFO:root:Training completed! Final loss: 0.0003


### Get the explanations

In [20]:
motif_subgraphs, graphlet_imp, edge_imp = get_explanations(
    model_name="DyGFormer",
    dataset_name=DATASET_NAME,
    test_data=test_data,
    examples=example_data,
    tempme=tempme,
    neighbor_finder=neighbor_finder,
    num_walks=NUM_WALKS,
)

100%|██████████| 5000/5000 [00:31<00:00, 156.48it/s]
INFO:root:Saved motif subgraphs to /Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/saved_explanations/DyGFormer/wikipedia/tempme/motif_subgraphs.csv
INFO:root:Saved graphlet importance to /Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/saved_explanations/DyGFormer/wikipedia/tempme/graphlet_imp.csv
INFO:root:Saved edge importance to /Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/saved_explanations/DyGFormer/wikipedia/tempme/edge_imp.csv
INFO:root:Creating plots for graphlet importance...
INFO:root:Saved graphlet importance plots to /Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/saved_explanations/DyGFormer/wikipedia/tempme/plots/graphlet_imp_plot.png
INFO:root:Creating plots for edge importance...
INFO:root:Saved edge importance plots to /Users/justinhoang/Documents/Reveal Research/DyGLib-Explainability/saved_explanations/DyGFormer/wikipedia/tempme/plots/edg

In [21]:
motif_subgraphs

,src,dst,bgd
0,"([[[8735 8884]], [[ 229 1222 500 6665]]], [[[...","([[[222 500]], [[8723 9186 8735 8884]]], [[[13...","([[[633 633]], [[8574 8574 8574 8574]]], [[[ 2..."
1,"([[[9215 9215]], [[ 443 443 443 7397]]], [[[...","([[[ 443 1067]], [[8505 9215 8512 8986]]], [[[...","([[[8443 8443]], [[290 290 54 312]]], [[[1140..."
2,"([[[9167 9167]], [[6161 1189 6174 1428]]], [[[...","([[[3149 2516]], [[8340 8340 9167 9167]]], [[[...","([[[8453 8453]], [[ 310 4179 310 310]]], [[[..."
3,"([[[8560 8560]], [[ 668 668 18 3129]]], [[[...","([[[ 25 4020]], [[8252 8252 8420 8467]]], [[[...","([[[9159 9159]], [[5969 5969 5969 5969]]], [[[..."
4,"([[[8722 8722]], [[1322 1322 1322 1322]]], [[[...","([[[1322 1322]], [[8750 8750 8750 8750]]], [[[...","([[[8986 8986]], [[ 69 936 697 2003]]], [[[..."
...,...,...,...
4995,"([[[8884 8735]], [[ 500 500 1498 5670]]], [[[...","([[[ 500 1398]], [[8884 8884 8624 8594]]], [[[...","([[[8639 8639]], [[1043 1043 843 3880]]], [[[..."
4996,"([[[8459 8470]], [[ 445 445 1570 1570]]], [[[...","([[[1067 69]], [[8284 8284 8589 8876]]], [[[...","([[[1097 1097]], [[8713 8713 8713 8713]]], [[[..."
4997,"([[[9147 9147]], [[ 0 0 5792 5792]]], [[[...","([[[5792 5792]], [[9147 9147 9147 9147]]], [[[...","([[[8638 8638]], [[2337 5328 211 211]]], [[[..."
4998,"([[[8577 8293]], [[1220 1278 3704 292]]], [[[...","([[[5880 3115]], [[ 0 0 8264 8332]]], [[[...","([[[8840 8840]], [[3312 4214 3312 3352]]], [[[..."


In [22]:
graphlet_imp

,src,dst,bgd
0,0.834391,0.813734,0.826800
1,0.803221,0.809780,0.828442
2,0.823023,0.814674,0.810674
3,0.800943,0.831272,0.844954
4,0.836734,0.830301,0.818989
...,...,...,...
4995,0.800883,0.810820,0.812738
4996,0.820812,0.806219,0.834706
4997,0.865726,0.838706,0.822877
4998,0.813146,0.827453,0.788300


In [23]:
edge_imp

,src,dst,bgd
0,0.000000,0.000000,0.428428
1,0.410052,0.415422,0.000000
2,0.000000,0.000000,0.823590
3,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.840414
...,...,...,...
4995,0.000000,0.000000,0.395064
4996,0.000000,0.000000,0.000000
4997,0.456231,0.000000,0.000000
4998,0.000000,0.000000,0.000000


### Check explanations

In [24]:
(
    fid_prob,
    fid_logit,
    expl_auc,
    expl_ap,
    expl_acc,
    base_auc,
    base_ap,
    base_acc
) = assess_explanations(
    test_data=test_data,
    examples=example_data,
    model=model,
    tempme=tempme,
    neighbor_finder=neighbor_finder,
    num_walks=NUM_WALKS,
)

  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [00:54<00:00, 90.93it/s] 


FIDELITY EVALUATION

Base Model (No Explanation):
  APS: 1.0000
  AUC: 1.0000
  ACC: 1.0000

With Explanations:
  APS: 0.9365
  AUC: 0.6335
  ACC: 0.8960

Fidelity Metrics:
  Fidelity Prob:  0.7847
  Fidelity Logit: 0.5075

INTERPRETATION:
Fidelity measures how much the explanation changes predictions
  Higher fidelity = explanation preserves original model behavior
  Lower fidelity = explanation significantly alters predictions

Fidelity prob: 0.7847
  Moderate: Some prediction changes with explanations

Fidelity logit: 0.5075
  High: Explanations preserve model predictions well
